# 02 From Fortran to JAX: read a routine, change it, check it

- **Tier:** laptop (CPU only)
- **Run time:** about 4 min (measured: 251 s on a compute node pinned to 8 cores; three runs of the gyre, mostly
  compiling)
- **Peak memory:** about 2 GB (measured: 2.0 GiB)
- **Experiment:** `verification/tutorial_barotropic_gyre`, variant `input`
- **Shows:** how a `.F` file reads in mitjax (Fortran-index arrays, loops, `# :NN` line citations, vertical loops
  as `scan_k` / `scan_levels`, the per-statement MAX/MIN convention); then one line of one routine changed, the
  model rerun and compared with the Fortran's `results/`.

Where to look things up:
- [`docs/fortran_map.md`](../docs/fortran_map.md): every MITgcm file and routine and the Python file that ports it,
  by directory, e.g. [`model/src`](../docs/fortran_map.md#modelsrc),
  [`pkg/mom_fluxform`](../docs/fortran_map.md#pkgmom_fluxform),
  [`pkg/generic_advdiff`](../docs/fortran_map.md#pkggeneric_advdiff),
  [`pkg/seaice`](../docs/fortran_map.md#pkgseaice); its first section is the short version of the conventions
  below.
- [`docs/READING_GUIDE.md`](../docs/READING_GUIDE.md): the long version, with worked examples (section numbers are
  quoted below as §n).
- [`docs/review/M1_SIDE_BY_SIDE.html`](../docs/review/M1_SIDE_BY_SIDE.html): the core routines of the dynamics
  side by side with their Fortran, line by line (open the file in a browser).

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import html
import inspect
from pathlib import Path

from IPython.display import HTML, display

import mitjax
from mitjax import paths

UPSTREAM = paths.UPSTREAM                          # $MJX_UPSTREAM: MITgcm at 63cdc0b
PKG = Path(mitjax.__file__).parent                 # the mitjax package


def fortran(rel, a, b):
    """Lines a..b of an MITgcm source file, numbered as in the file."""
    lines = (UPSTREAM / rel).read_text().splitlines()
    return "\n".join(f"{n:4d}  {lines[n - 1]}" for n in range(a, b + 1))


def python(rel, start, stop=None):
    """The lines of mitjax/<rel> from the first line containing `start` up to (not including) the first after it
    containing `stop` (to the end of the file without `stop`)."""
    lines = (PKG / rel).read_text().splitlines()
    i = next(n for n, ln in enumerate(lines) if start in ln)
    j = next((n for n in range(i + 1, len(lines)) if stop and stop in lines[n]), len(lines))
    return "\n".join(lines[i:j]).rstrip()


def side_by_side(left_title, left, right_title, right):
    cell = '<td style="vertical-align:top;text-align:left"><b>{}</b><pre style="font-size:11px">{}</pre></td>'
    display(HTML("<table><tr>" + cell.format(html.escape(left_title), html.escape(left))
                 + cell.format(html.escape(right_title), html.escape(right)) + "</tr></table>"))

## One routine: `MOM_U_XVISCFLUX`

`pkg/mom_fluxform/mom_u_xviscflux.F` computes the zonal viscous flux of `u` at one level. Its port is
`mitjax/pkg/mom_fluxform/mom_u_xviscflux.py` (one `.py` per `.F`, same path, lower-case names).

In [2]:
side_by_side("pkg/mom_fluxform/mom_u_xviscflux.F", fortran("pkg/mom_fluxform/mom_u_xviscflux.F", 52, 71),
             "mitjax/pkg/mom_fluxform/mom_u_xviscflux.py",
             python("pkg/mom_fluxform/mom_u_xviscflux.py", "def mom_u_xviscflux"))

"pkg/mom_fluxform/mom_u_xviscflux.F 52 53 C - Laplacian and bi-harmonic terms 54 DO j=1-OLy,sNy+OLy-1 55 DO i=1-OLx,sNx+OLx-1 56 xViscFluxU(i,j) = 57 & _dyF(i,j,bi,bj)*drF(k)*_hFacC(i,j,k,bi,bj) 58 & *( 59 & -viscAh_D(i,j)*( uFld(i+1,j)-uFld(i,j) ) 60 & *cosFacU(j,bi,bj) 61 & +viscA4_D(i,j)*(del2u(i+1,j)-del2u(i,j)) 62 #ifdef COSINEMETH_III 63 & *sqCosFacU(j,bi,bj) 64 #else 65 & *cosFacU(j,bi,bj) 66 #endif 67 & )*_recip_dxF(i,j,bi,bj) 68 c & *deepFacC(k) ! dyF scaling factor 69 c & *recip_deepFacC(k) ! recip_dxF scaling factor 70 ENDDO 71 ENDDO","mitjax/pkg/mom_fluxform/mom_u_xviscflux.pydef mom_u_xviscflux(k, uFld, del2u, xViscFluxU, viscAh_D, viscA4_D, *, cfg, grid): """"""MOM_U_XVISCFLUX(bi,bj,k, uFld, del2u, xViscFluxU, viscAh_D, viscA4_D, myThid) @63cdc0b pkg/mom_fluxform/mom_u_xviscflux.F:6-74 C Calculates the area integrated zonal viscous fluxes of U: C F^x = - \\frac{ \\Delta y_f \\Delta r_f h_c }{\\Delta x_f} C ( A_h \\delta_i u - A_4 \\delta_i \\nabla^2 u ) C k :: vertical level C uFld :: zonal flow C del2u :: Laplacian of zonal flow C xViscFluxU :: viscous fluxes Returns xViscFluxU. #ifdef COSINEMETH_III (:62-66) follows the build (defined in every M1 build). `_dyF`, `_hFacC`, `_recip_dxF` are dyF, hFacC, recip_dxF. The point loop runs on the whole (i,j) range. """""" sNx, sNy, OLx, OLy = cfg.size.sNx, cfg.size.sNy, cfg.size.OLx, cfg.size.OLy dyF, drF, hFacC, recip_dxF = grid.dyF, grid.drF, grid.hFacC, grid.recip_dxF cosFacU = grid.cosFacU cF4 = grid.sqCosFacU if cfg.cpp.flag(""COSINEMETH_III"", _opt(cfg)) else grid.cosFacU # :62-66 # - Laplacian and bi-harmonic terms j = loop_j(1-OLy, sNy+OLy-1) # :54-71 i = loop_i(1-OLx, sNx+OLx-1) xViscFluxU = xViscFluxU.at[i, j].set( dyF[i, j]*drF[k]*hFacC[i, j, k] * ( -viscAh_D[i, j]*(uFld[i+1, j]-uFld[i, j]) * cosFacU[j] + viscA4_D[i, j]*(del2u[i+1, j]-del2u[i, j]) * cF4[j] )*recip_dxF[i, j]) return xViscFluxU"


How to read it (READING_GUIDE §1-4):

- **The docstring head** names the Fortran call and the span it ports: `@63cdc0b
  pkg/mom_fluxform/mom_u_xviscflux.F:6-74`. In the body, `# :54-71` points at lines of that same file.
- **Arguments** are the Fortran's in the Fortran order, minus `bi, bj, myThid`; outputs are returned. The common
  blocks come in as objects: `grid` is `GRID.h`, `cfg.size` is `SIZE.h`, `cfg.cpp` the preprocessed CPP options.
- **Fortran-index arrays.** `uFld[i+1, j]` is `uFld(i+1,j)` with Fortran index values: every array keeps its
  declared bounds (`1-OLx:sNx+OLx`), and an index outside them is an error.
- **Loops.** `j = loop_j(1-OLy, sNy+OLy-1)` is the `DO j` statement and `.at[i, j].set(...)` the assignment
  inside the loops: the whole loop nest at once. The tile loop `DO bj / DO bi` is implicit (every statement acts
  on all tiles; storage is `[tile, k, j, i]`).
- **CPP options** are Python `if`s on `cfg.cpp` (`#ifdef COSINEMETH_III`); integer, logical and character
  namelist values are static, REAL values are traced (they can be differentiated).
- **Same statements, same order, same parentheses.** Floating-point addition is not associative; the port keeps
  the Fortran's expression tree so that it reproduces the Fortran to the last bit.

## Vertical loops: `scan_k` and `scan_levels`

A recursion in `k` (a tridiagonal solve, an integration from the surface) is written with `scan_k`
(READING_GUIDE §7). Here the forward sweep of `model/src/solve_tridiagonal.F`, used by the implicit vertical
diffusion: `k = 1` is written out, then `scan_k` runs `k = 2..Nr` in the Fortran order; its carry `(cpm1, ypm1)`
is level `k-1`, `x[...]` the arrays at level `k`. A pivot test `IF (tmpVar .NE. 0)` becomes `jnp.where` with a
guarded division (`safe_div`), so that the branch not taken stays finite for the derivative (READING_GUIDE §6).

In [3]:
side_by_side("model/src/solve_tridiagonal.F", fortran("model/src/solve_tridiagonal.F", 237, 275),
             "mitjax/model/src/solve_tridiagonal.py",
             python("model/src/solve_tridiagonal.py", "# :241-254", "# :279-295"))

"model/src/solve_tridiagonal.F 237 CADJ loop = sequential 238 C-- Forward sweep 239 DO k=1,Nr 240 241 IF ( k .EQ. 1 ) THEN 242 DO j=1-OLy,sNy+OLy 243 DO i=1-OLx,sNx+OLx 244 IF ( b3d(i,j,1).NE.0. _d 0 ) THEN 245 recVar = 1. _d 0 / b3d(i,j,1) 246 c3d_prime(i,j,1) = c3d(i,j,1)*recVar 247 y3d_prime(i,j,1) = y3d_m1(i,j,1)*recVar 248 ELSE 249 c3d_prime(i,j,1) = 0. _d 0 250 y3d_prime(i,j,1) = 0. _d 0 251 errCode = 1 252 ENDIF 253 ENDDO 254 ENDDO 255 ELSE 256 DO j=1-OLy,sNy+OLy 257 DO i=1-OLx,sNx+OLx 258 tmpVar = b3d(i,j,k) - a3d(i,j,k)*c3d_prime(i,j,k-1) 259 IF ( tmpVar .NE. 0. _d 0 ) THEN 260 recVar = 1. _d 0 / tmpVar 261 c3d_prime(i,j,k) = c3d(i,j,k)*recVar 262 y3d_prime(i,j,k) = ( y3d_m1(i,j,k) 263 & - a3d(i,j,k)*y3d_prime(i,j,k-1) 264 & )*recVar 265 ELSE 266 c3d_prime(i,j,k) = 0. _d 0 267 y3d_prime(i,j,k) = 0. _d 0 268 errCode = 1 269 ENDIF 270 ENDDO 271 ENDDO 272 ENDIF 273 274 C- end k-loop 275 ENDDO","mitjax/model/src/solve_tridiagonal.py # :241-254 forward sweep, k = 1 nz = b3d[i, j, 1] != 0.0 # IF ( b3d(i,j,1).NE.0. _d 0 ) THEN recVar = safe_div(1.0, b3d[i, j, 1], nz) # recVar = 1. _d 0 / b3d(i,j,1) c3d_prime = c3d_prime.at[i, j, 1].set(jnp.where(nz, c3d[i, j, 1]*recVar, 0.0)) y3d_prime = y3d_prime.at[i, j, 1].set(jnp.where(nz, y3d_m1[i, j, 1]*recVar, 0.0)) errCode = jnp.where(jnp.any(~nz, axis=(1, 2)), 1, errCode) # :255-275 forward sweep, k = 2..Nr def forward(carry, x): cpm1, ypm1 = carry tmpVar = x[""b""][i, j] - x[""a""][i, j]*cpm1[i, j] # :258 nzk = tmpVar != 0.0 # :259 rv = safe_div(1.0, tmpVar, nzk) # :260 cpk = x[""cp""].at[i, j].set(jnp.where(nzk, x[""c""][i, j]*rv, 0.0)) # :261 / :266 ypk = x[""yp""].at[i, j].set(jnp.where(nzk, (x[""y""][i, j] # :262-264 / :267 - x[""a""][i, j]*ypm1[i, j] )*rv, 0.0)) return (cpk, ypk), {""cp"": cpk, ""yp"": ypk, ""err"": jnp.any(~nzk, axis=(1, 2))} _, fw = scan_k(forward, (level(c3d_prime, 1), level(y3d_prime, 1)), range(2, Nr+1), lambda k: {""a"": level(a3d, k), ""b"": level(b3d, k), ""c"": level(c3d, k), ""y"": level(y3d_m1, k), ""cp"": level(c3d_prime, k), ""yp"": level(y3d_prime, k)}) c3d_prime = set_levels(c3d_prime, fw[""cp""], range(2, Nr+1)) y3d_prime = set_levels(y3d_prime, fw[""yp""], range(2, Nr+1)) errCode = jnp.where(jnp.any(fw[""err""], axis=0), 1, errCode) # :268 errCode = 1"


A routine the Fortran calls once per level (`DO k ... CALL INTEGRATE_FOR_W(..k..)`) is written for one level,
and its caller runs it with `scan_levels`; levels the Fortran treats differently (here `k = 1` and `k = Nr`) run
as separate calls next to the scan (`peel`):

In [4]:
side_by_side("model/src/integr_continuity.F", fortran("model/src/integr_continuity.F", 266, 281),
             "mitjax/model/src/integr_continuity.py",
             python("model/src/integr_continuity.py", "# :267  DO k=Nr,1,-1", "if params.implicitIntGravWave"))

"model/src/integr_continuity.F 266 267 DO k=Nr,1,-1 268 C-- Integrate continuity vertically for vertical velocity 269 270 CALL INTEGRATE_FOR_W( 271 I bi, bj, k, uFld, vFld, 272 I addMass, rStarDhDt, 273 O wVel, 274 I myIter, myThid ) 275 276 IF ( k.EQ.Nr .AND. myIter.NE.0 .AND. usingPCoords 277 & .AND. fluidIsWater .AND. useRealFreshWaterFlux ) THEN 278 279 DO j=1,sNy 280 DO i=1,sNx 281 wVel(i,j,k,bi,bj) = wVel(i,j,k,bi,bj)","mitjax/model/src/integr_continuity.py # :267 DO k=Nr,1,-1 as a level scan (KERNEL_GUIDE §4): k = Nr and k = 1 static (INTEGRATE_FOR_W's branches) def level_k(k, wVel): wVel = integrate_for_w(k, uFld, vFld, addMass, rStarDhDt, wVel, myIter, # :270-274 cfg=cfg, grid=grid, params=params) if k == sz.Nr and params.usingPCoords and params.fluidIsWater and params.useRealFreshWaterFlux: raise NotImplementedError(""INTEGR_CONTINUITY: :276-286 (p coordinates, real fresh water) is not ported"") return wVel from mitjax.ops.scan_k import scan_levels wVel = scan_levels(level_k, wVel, 1, sz.Nr, down=True, peel=(1, 1))"


## MAX and MIN: one convention per statement

Every Fortran `MAX`/`MIN` of REAL values is `MAX(a, b, p="a")` or `p="b"` (READING_GUIDE §9). `p` names the
argument gfortran returns at that statement when the two tie with different signs of zero (`+0` vs `-0`) or one
is NaN; it was measured, statement by statement, from our gfortran builds. It matters only for bitwise agreement
(and for the derivative at a tie). The lopping factor `hFacC` of `model/src/ini_masks_etc.F`:

In [5]:
side_by_side("model/src/ini_masks_etc.F", fortran("model/src/ini_masks_etc.F", 105, 121),
             "mitjax/model/src/ini_masks_etc.py",
             python("model/src/ini_masks_etc.py", "# :105-124", "# :126-143"))

"model/src/ini_masks_etc.F 105 C-- Calculate lopping factor hFacC : over-estimate the part inside of the domain 106 C taking into account the lower_R Boundary (Bathymetry / Top of Atmos) 107 DO k=1, Nr 108 hFacMnSz = MAX( hFacMin, MIN(hFacMinDr*recip_drF(k),oneRL) ) 109 DO j=1-OLy,sNy+OLy 110 DO i=1-OLx,sNx+OLx 111 C o Non-dimensional distance between grid bound. and domain lower_R bound. 112 hFac_loc = (rF(k)-R_low(i,j,bi,bj))*recip_drF(k) 113 C o Select between, closed, open or partial (0,1,0-1) 114 hFac_loc = MIN( MAX( hFac_loc, zeroRL ) , oneRL ) 115 C o Impose minimum fraction and/or size (dimensional) 116 IF ( hFac_loc.LT.hFacMnSz*halfRL .OR. 117 & R_low(i,j,bi,bj).GE.Ro_surf(i,j,bi,bj) ) THEN 118 hFacC(i,j,k,bi,bj) = zeroRS 119 ELSE 120 hFacC(i,j,k,bi,bj) = MAX( hFac_loc, hFacMnSz ) 121 ENDIF","mitjax/model/src/ini_masks_etc.py # :105-124 lopping factor hFacC: over-estimate the part inside of the domain (lower_R boundary) k, j, i = loops_kji((1, Nr), (1-OLy, sNy+OLy), (1-OLx, sNx+OLx)) hFacMnSz = MAX(hFacMin, MIN(hFacMinDr*recip_drF[k], oneRL, p=""b""), p=""b"") # :108 hFac_loc = (rF[k]-R_low[i, j])*recip_drF[k] hFac_loc = MIN(MAX(hFac_loc, zeroRL, p=""b""), oneRL, p=""a"") # :114 hFacC = hFacC.at[i, j, k].set(jnp.where((hFac_loc < hFacMnSz*halfRL) | (R_low[i, j] >= Ro_surf[i, j]), zeroRS, MAX(hFac_loc, hFacMnSz, p=""b""))) # :120"


At a few statements gfortran's choice depends on the build (the options and `SIZE.h`). There mitjax looks your
build up by a hash of its preprocessed options and `SIZE.h`; where our Fortran reference never measured the
statement in your build, mitjax takes a documented default and prints one warning naming the statement. Notebook
04 shows it.

## Change one line and check it against the Fortran

Suppose you want to try a change in `MOM_U_XVISCFLUX`, in its Laplacian term `-viscAh_D*(uFld(i+1,j)-uFld(i,j))`.
We try two changes of that one line:

- **reordered:** `-viscAh_D*uFld(i+1,j) + viscAh_D*uFld(i,j)`, the same mathematics with a different order of
  floating-point operations;
- **halved:** `-0.5*viscAh_D*(uFld(i+1,j)-uFld(i,j))`, half the zonal Laplacian flux of `u`: a real change.

In your own copy of mitjax you would edit `mitjax/pkg/mom_fluxform/mom_u_xviscflux.py`. In this notebook we leave
the files alone and replace the function where its caller looks it up: `mom_fluxform.py` imported it by name, so
its module attribute `mom_u_xviscflux` is what the model calls when the time step is traced.

In [6]:
import jax
import numpy as np

from mitjax.farray import loop_i, loop_j
import mitjax.pkg.mom_fluxform.mom_fluxform as mom_fluxform

original = mom_fluxform.mom_u_xviscflux


def changed_xviscflux(laplacian):
    """MOM_U_XVISCFLUX as in mitjax, with the Laplacian term (the one line marked CHANGED) given by `laplacian`."""
    def mom_u_xviscflux(k, uFld, del2u, xViscFluxU, viscAh_D, viscA4_D, *, cfg, grid):
        sNx, sNy, OLx, OLy = cfg.size.sNx, cfg.size.sNy, cfg.size.OLx, cfg.size.OLy
        dyF, drF, hFacC, recip_dxF = grid.dyF, grid.drF, grid.hFacC, grid.recip_dxF
        cosFacU = grid.cosFacU
        cF4 = grid.sqCosFacU if cfg.cpp.flag("COSINEMETH_III", "MOM_COMMON_OPTIONS.h") else grid.cosFacU
        j = loop_j(1-OLy, sNy+OLy-1)
        i = loop_i(1-OLx, sNx+OLx-1)
        xViscFluxU = xViscFluxU.at[i, j].set(
            dyF[i, j]*drF[k]*hFacC[i, j, k]
            * (
               laplacian(viscAh_D[i, j], uFld[i+1, j], uFld[i, j])     # CHANGED: -viscAh_D*(u(i+1)-u(i))
               * cosFacU[j]
               + viscA4_D[i, j]*(del2u[i+1, j]-del2u[i, j])
               * cF4[j]
              )*recip_dxF[i, j])
        return xViscFluxU
    return mom_u_xviscflux


CHANGES = {
    "reordered": lambda A, u_ip1, u_i: -A*u_ip1 + A*u_i,
    "halved": lambda A, u_ip1, u_i: -0.5*A*(u_ip1 - u_i),
}


def new_out(name):
    n = 1
    while Path(f"runs/{name}-{n}").exists():
        n += 1
    return Path(f"runs/{name}-{n}")


VERIFICATION = UPSTREAM / "verification"
exp = mitjax.load(VERIFICATION / "tutorial_barotropic_gyre", variant="input")
assert exp.config.cfg.cpp.flag("ALLOW_MOM_COMMON", "MOM_FLUXFORM_OPTIONS.h")   # so the options header above is right

runs = {"original": exp.run(out=new_out("02_gyre"))}
for name, laplacian in CHANGES.items():
    mom_fluxform.mom_u_xviscflux = changed_xviscflux(laplacian)
    jax.clear_caches()                                          # trace the time step again, with the change
    try:
        runs[name] = exp.run(out=new_out(f"02_gyre_{name}"))
    finally:
        mom_fluxform.mom_u_xviscflux = original                 # put the original back
        jax.clear_caches()

In [7]:
reports = {name: mitjax.compare(run, exp.results()) for name, run in runs.items()}
for name, rep in reports.items():
    print(f"{name:10s}", rep.summary)
print("\nvariable   " + "".join(f"{n:>11s}" for n in reports))
for k, var in enumerate(reports["original"].run.variables):
    print(f"{var.name:10s} " + "".join(f"{rep.run.variables[k].digits:11d}" for rep in reports.values()))
eta0 = runs["original"].fields["etaN"]
for name in CHANGES:
    print(f"max |etaN {name} - etaN original| = {np.abs(runs[name].fields['etaN'] - eta0).max():.2e} m "
          f"(|etaN| <= {np.abs(eta0).max():.2e} m)")

original   Y Y Y Y>16<16 16 16 22 16 16 16 22 16 16 16 16 16 16 16 16  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . pass  tutorial_barotropic_gyre
reordered  Y Y Y Y>14<16 16 16 22 16 16 16 22 16 16 11 16 16 16 12 16  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . pass  tutorial_barotropic_gyre
halved     Y Y Y Y> 5<16 16 16 22 16 16 16 22  5  5  0  5  5  5  5  5  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . FAIL  tutorial_barotropic_gyre

variable      original  reordered     halved
PS                  16         14          5
Tmn                 16         16         16
Tmx                 16         16         16
Tav                 16         16         16
Tsd                 22         22         22
Smn                 16         16         16
Smx                 16         16         16
Sav                 16         16         16
Ssd                 22         22         22
Umn                 16         16          5
Umx                 1

What this says:

- The original code reproduces the Fortran's `results/` to every printed digit (16; 22 is testreport's count for
  two zeros).
- The reordering changes the last digits after ten time steps: testreport still passes (its criterion is 10
  digits), but the result is no longer the Fortran's bit for bit. That is why the port keeps the Fortran's
  statements, order and parentheses, and why its own tests compare with the Fortran bit for bit on our reference
  machine: there a harmless reordering and a real error are not confused.
- Halving the flux fails testreport: the velocity and surface-height statistics keep about 5 digits.

The asserts below hold on any platform (plan decision 10: another CPU may round differently, so nothing here is
asserted bit for bit).

In [8]:
MIN_DIGITS = 10
for name in ("original", "reordered"):
    assert reports[name].verdict == "pass", name
    assert reports[name].run.variables[0].digits >= MIN_DIGITS, name
assert reports["halved"].verdict == "FAIL"
assert np.abs(runs["halved"].fields["etaN"] - eta0).max() > 0